# LM Cost All-in-One Precompute Runner

This notebook runs the consolidated pipeline script to generate all preprocessed artifacts for the app.

In [ ]:
from pathlib import Path
import subprocess
import sys
import json

cwd = Path.cwd()
if cwd.name == "lm_cost" and cwd.parent.name == "apps":
    app_dir = cwd
    repo_root = cwd.parent.parent
else:
    repo_root = cwd
    app_dir = repo_root / "apps" / "lm_cost"
data_dir = app_dir / "data"
script_path = app_dir / "precompute_lm_cost_all.py"

print(f"Repo root: {repo_root}")
print(f"Script: {script_path}")
print(f"Data dir: {data_dir}")

if not script_path.exists():
    raise FileNotFoundError(f"Missing script: {script_path}")
if not data_dir.exists():
    raise FileNotFoundError(f"Missing data folder: {data_dir}")

In [ ]:
# Default input expected by the consolidated runner
input_path = data_dir / "LM_CS_slim.csv.gz"

if not input_path.exists():
    raise FileNotFoundError(f"Missing input source file: {input_path}")

cmd = [
    sys.executable,
    str(script_path),
    "--input", str(input_path),
    "--outdir", str(data_dir),
]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"Precompute failed with exit code {result.returncode}")

expected = [
    data_dir / "LM_CS_slim_prepared.pkl",
    data_dir / "LM_CS_slim_prepared.parquet",
    data_dir / "LM_CS_slim_preprocess_manifest.json",
    data_dir / "LM_CS_scorecard_default.pkl",
    data_dir / "LM_CS_scorecard_default_manifest.json",
]

for path in expected:
    print(path.name, "=>", "FOUND" if path.exists() else "NOT FOUND")

manifest_path = data_dir / "LM_CS_scorecard_default_manifest.json"
if manifest_path.exists():
    payload = json.loads(manifest_path.read_text(encoding="utf-8"))
    print("\nScorecard manifest summary:")
    print("generated_at_utc:", payload.get("generated_at_utc"))
    print("groups:", payload.get("groups"))
    print("scorecard_rows:", payload.get("scorecard_rows"))